# NJ148 on Colab (L4): Italian BERT baseline, session B

Token classification with BIO labels, `dbmdz/bert-base-italian-xxl-cased` (public, no login needed), config
`configs/encoder_bert_it.yaml`. Two arms on the same folds and training examples as the LLM: E1 data (gold only) and
E3 data (gold + style-anchored synthetic + negatives). Each fold trains and then labels its held-out notes in one step.
Needs `MyDrive/nj148/nj148_transfer_e3x_bert.tar.gz`. Results go to `MyDrive/nj148/runs_bert/`. Outputs show counts, hashes and pass/fail only.

**Run cells 1, 2, 3, 3b (tiny BERT smoke test), then 7.** The last cell releases the server.
If the session drops, run 1, 2, 3 and 7 again: finished folds are skipped.


In [ ]:
# 1. GPU and Drive
import os, pathlib
# Colab ships TensorFlow; keep transformers from importing it (its 'no cuda drivers' message is about TensorFlow, not torch)
os.environ['USE_TF'] = '0'; os.environ['TRANSFORMERS_NO_TF'] = '1'; os.environ['PYTHONUNBUFFERED'] = '1'
import torch
assert torch.cuda.is_available(), 'no GPU: the server must be an L4'
print(torch.cuda.get_device_name(0), 'bf16', torch.cuda.is_bf16_supported())
from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/nj148')
RUNS = DRIVE / 'runs_bert'   # results of this experiment only
for sub in ('checkpoints', 'predictions_secure', 'logs'):
    (RUNS / sub).mkdir(parents=True, exist_ok=True)
print('archive present:', (DRIVE / 'nj148_transfer_e3x_bert.tar.gz').exists())

In [ ]:
print((DRIVE / 'nj148_transfer_e3x_bert.tar.gz').stat().st_size)   # должно быть 5859675

In [ ]:
# 2. Unpack, clone the committed code from the bundle, place gold and synthetic data
import subprocess, hashlib, json, shutil
WORK, REPO = pathlib.Path('/content/work_bert'), pathlib.Path('/content/repo_bert')

sh = lambda cmd: print(subprocess.run(cmd, shell=True, capture_output=True, text=True, check=True).stdout.strip())
shutil.rmtree(WORK, ignore_errors=True); shutil.rmtree(REPO, ignore_errors=True); WORK.mkdir()
sh(f'tar -xzf {DRIVE}/nj148_transfer_e3x_bert.tar.gz -C {WORK}')
manifest = json.loads((WORK / 'transfer_manifest.json').read_text())
sh(f'git clone -q -b research-reconstruction {WORK}/repo.bundle {REPO}')
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == manifest['commit'], f"commit mismatch {head} vs {manifest['commit']}"
(REPO / 'data_secure' / 'gold').mkdir(parents=True, exist_ok=True)
shutil.copy(WORK / 'gold_v2.jsonl', REPO / 'data_secure' / 'gold' / 'gold_v2.jsonl')
sha = hashlib.sha256((REPO / 'data_secure/gold/gold_v2.jsonl').read_bytes()).hexdigest()
assert sha == manifest['gold_sha256'], 'gold hash mismatch'
sh(f'cp -R {WORK}/synthetic_data/. {REPO}/synthetic_data/')
bad = [p for p, h in manifest['files'].items() if hashlib.sha256((REPO / p).read_bytes()).hexdigest() != h]
assert not bad, f'{len(bad)} files differ from the manifest'
# Official outputs must be real folders inside the repo: the code resolves paths, so a symlink to Drive would turn an
# official run into a development run. Results are copied to Drive after each step and restored from it here.
def backup():
    for sub in ('checkpoints', 'predictions_secure'):
        (REPO / sub).mkdir(exist_ok=True)
        sh(f'rsync -a {REPO}/{sub}/ {RUNS}/{sub}/')
    print('backed up to Drive')
def restore():
    for sub in ('checkpoints', 'predictions_secure'):
        (REPO / sub).mkdir(exist_ok=True)
        sh(f'rsync -a {RUNS}/{sub}/ {REPO}/{sub}/')
# macOS metadata files ('._*') would make the tree look modified and block official runs
sh(f"find {REPO} -name '._*' -print -delete | wc -l")
import sys; sys.path.insert(0, str(REPO))
from src.infrastructure.provenance import check_source_integrity
restore()
integrity = check_source_integrity(REPO)
print('tree clean for official runs:', integrity.clean, '' if integrity.clean else integrity.summary())
print('commit', head[:7], '| gold sha', sha[:8], '| files verified', len(manifest['files']))

In [ ]:
# 3. Packages pinned to the versions the pipeline was tested with (local smoke test). transformers 5.x is NOT used.
# torch stays Colab's (no torch 2.2.2 build exists for Python 3.13 / CUDA 13); bitsandbytes 0.50.2 passed NF4 on this L4.
PINS = {"transformers": "4.46.3", "peft": "0.21.1", "accelerate": "1.14.0", "tokenizers": "0.20.3", "safetensors": "0.8.0", "huggingface_hub": "0.36.2", "bitsandbytes": "0.50.2"}
spec = ' '.join(f'{k}=={v}' for k, v in PINS.items())
!pip -q uninstall -y torchao   # peft refuses Colab's old torchao 0.10; the tested environment had no torchao
!pip -q install {spec} sentencepiece PyYAML scikit-learn pytest 2>&1 | tail -3
import importlib.metadata as md
got = {k: md.version(k) for k in PINS}
print(got, '| torch', md.version('torch'))
wrong = {k: (got[k], v) for k, v in PINS.items() if got[k] != v}
assert not wrong, f'version mismatch (installed, pinned): {wrong}'
import importlib.util
assert importlib.util.find_spec('torchao') is None, 'torchao is still installed'
!pip freeze > /content/drive/MyDrive/nj148/runs_bert/logs/requirements-training.colab.lock
print('pins ok; lock written to Drive')

In [ ]:
# 3b. The encoder smoke test that passed locally: tiny random BERT, invented data, no project data, no download
import subprocess
r = subprocess.run(['python', 'scripts/smoke_tiny_encoder.py'], cwd=REPO, capture_output=True, text=True)
print('exit', r.returncode)
print((r.stdout + r.stderr)[-3000:])

In [ ]:
# 7. OFFICIAL BERT runs: train and label held-out notes per fold, then back up to Drive. Finished folds are skipped.
# Logs: MyDrive/nj148/runs_bert/logs/
JOBS = [(e, k) for e in ('E1', 'E3') for k in range(1, 6)]
SEED = 42
import glob, shutil, subprocess, time
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert check_source_integrity(REPO).clean, 'tree not clean: official runs would be refused'
for exp, fold in JOBS:
    pred = glob.glob(str(REPO / 'predictions_secure' / f'{exp}-BERT__*__seed{SEED}' / f'fold_{fold}'))
    if any(pathlib.Path(p, 'prediction_manifest.json').exists() for p in pred):
        print(exp, fold, 'already done'); continue
    ckpt = REPO / 'checkpoints' / 'official' / 'bert_it' / exp / f'fold_{fold}' / f'seed_{SEED}'
    for partial in [str(ckpt)] * ckpt.exists() + pred:   # an interrupted fold is moved aside, never reused
        dest = RUNS / 'logs' / 'failed_partial' / f'{exp}_fold{fold}_{int(time.time())}_{pathlib.Path(partial).parent.name}'
        dest.parent.mkdir(parents=True, exist_ok=True); shutil.move(partial, dest)
        print(exp, fold, 'moved an unfinished folder aside')
    log = RUNS / 'logs' / f'{exp}-BERT_fold{fold}_seed{SEED}.log'
    t0 = time.time()
    with open(log, 'a') as f:
        r = subprocess.run(['python', '-u', '-m', 'src.encoder.run', '--experiment', exp, '--fold', str(fold), '--seed', str(SEED), '--official'],
                           cwd=REPO, stdout=f, stderr=subprocess.STDOUT, text=True)
    print(exp, fold, 'exit', r.returncode, '| minutes', round((time.time() - t0) / 60, 1))
    if r.returncode:
        print(open(log).read()[-1500:]); break
    backup()


In [ ]:
try:
    from google.colab import runtime
    runtime.unassign()
    print('runtime released')
except Exception as e:
    print('could not release from VS Code:', type(e).__name__, '- remove the server manually: Colab: Remove Server')
